# Spark Connect + Armada Demo

**Architecture:**
- Spark driver + Connect server run **in the cluster** (Armada cluster mode).
- This notebook is a **thin gRPC client** — no JVM locally.

**Setup:**
```bash
./scripts/runJupyter.sh -C
```

`runJupyter.sh` parses the driver job ID from `spark-submit`'s output and
exports `SPARK_CONNECT_HOST` / `SPARK_CONNECT_PORT` / `SPARK_CONNECT_TOKEN`
into the Jupyter container — no port-forward needed. The hostname follows
Armada's Ingress template:

```
driver-<spark.connect.grpc.binding.port>-armada-<jobid>-0.<namespace>.<SPARK_CONNECT_INGRESS_DOMAIN>
```

`SPARK_CONNECT_INGRESS_DOMAIN` and the namespace come from your
`scripts/config.sh`; the rest is built per-job. The hostname resolves (via
a DNS-Only A record) to wherever your cluster's ingress controller is
reachable, where TLS terminates with the appropriate cert. See
`plans/spark-connect/auth.md` for the end-to-end topology used in this
repo's reference setup.

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# runJupyter.sh sets these per-job from spark-submit's output:
#   HOST = driver-<port>-armada-<jobid>-0.<namespace>.<SPARK_CONNECT_INGRESS_DOMAIN>
#   PORT = 443                          (TLS at the cluster ingress controller)
# Defaults below are fallbacks if you start Jupyter manually.
HOST = os.environ.get("SPARK_CONNECT_HOST", "host.docker.internal")
PORT = os.environ.get("SPARK_CONNECT_PORT", "443")
print(f"Spark Connect endpoint: sc://{HOST}:{PORT}")

# pyspark Connect implicitly forces use_ssl=True whenever ;token=... is in the
# URL (it refuses to send a bearer token over plaintext). That matches our
# setup: the cluster ingress controller terminates TLS, then forwards plain
# h2c to the driver pod's Spark Connect server.
TOKEN = os.environ.get("SPARK_CONNECT_TOKEN", "")
if TOKEN:
    CONNECT_URL = f"sc://{HOST}:{PORT}/;token={TOKEN}"
else:
    CONNECT_URL = f"sc://{HOST}:{PORT}"

spark = SparkSession.builder.remote(CONNECT_URL).getOrCreate()
print(f"Connected! Spark version: {spark.version}")

In [ ]:
spark.sql("SELECT 'Hello from Spark Connect on Armada!' as message").show(truncate=False)

## Dynamic Allocation Test

200 partitions with 10s delay each to trigger executor scale-up.

Watch with: `kubectl get pods -n default -w`

In [ ]:
import time

num_partitions = 200

print(f"Running Spark Pi with {num_partitions} slow partitions...")

def slow_pi_partition(iterator):
    import pandas as pd
    import time
    import random
    for pdf in iterator:
        time.sleep(10)
        count = 0
        total = len(pdf)
        for _ in range(total):
            x, y = random.random(), random.random()
            if x * x + y * y < 1:
                count += 1
        yield pd.DataFrame({"count": [count], "total": [total]})

start = time.time()

df_input = spark.range(0, 1000000, numPartitions=num_partitions)
df_result = df_input.mapInPandas(slow_pi_partition, schema="count long, total long")
totals = df_result.agg(
    F.sum("count").alias("count"),
    F.sum("total").alias("total")
).collect()

pi = 4.0 * totals[0]["count"] / totals[0]["total"]
elapsed = time.time() - start

print(f"Pi is approximately: {pi}")
print(f"Completed in {elapsed:.1f}s")

In [ ]:
spark.stop()
print("Disconnected")